In [1]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})


question = questions.iloc[0]["question"]
print("Question: ", question)
print("\n\nAbstract: ", documents.iloc[0].abstract)


Question:  Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?


Abstract:  Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells 

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


agent = ChatOpenAI(model="gpt-4o-mini", temperature=0, use_responses_api=True)
response = agent.invoke([("system", "Answer the medical question clearly and concisely."), ("human", question)])

print("\n\nQuestion:", question)
print("Answer:", response.content)



/Users/filipn/Documents/dev/WASP-DL-NLP/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Answer: [{'type': 'text', 'text': "Yes, mitochondria play a significant role in the remodeling of lace plant leaves during programmed cell death (PCD). They are involved in energy production and the regulation of apoptotic processes. During PCD, mitochondria can release factors that promote cell death and influence the degradation of cellular components, contributing to the structural changes observed in lace plant leaves. This process is essential for the plant's development and adaptation.", 'annotations': [], 'id': 'msg_0b47251f59fc4fd2006a84809b9e3087d180a1935a6741600a'}]


In [3]:
#### EMBEDDINGS
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np


embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", encode_kwargs={"normalize_embeddings": True})


text = "Mitochondria participate in programmed cell death."
embedding = embedding_model.embed_query(text)
print(type(embedding))
print(np.asarray(embedding).shape)



# query_result = embeddings.embed_query("This is a test document.")
# doc_result = embeddings.embed_documents(["This is a test document."])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8790.05it/s]


<class 'list'>
(384,)


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, length_function=len, is_separator_regex=False)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas) # type: ignore

print(texts[0])
print(texts[1])

page_content='Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has' metadata={'id': 21645374}
page_content='vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in e

In [5]:
from langchain_chroma import Chroma


vectordb = Chroma.from_documents(documents=texts, embedding=embedding_model, collection_name="pubmedqa",collection_metadata={"hnsw:space": "cosine"})


results = vectordb.similarity_search_with_score("What is programmed cell death?", k=3)
for document, score in results:
    print(f"Score: {score:.3f}")
    print(f"Document ID: {document.metadata['id']}")
    print(f"Text: {document.page_content}")
    print()

Score: 0.381
Document ID: 21645374
Text: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has

Score: 0.611
Document ID: 15223779
Text: a high phosphorylation level in serum-depleted uveal melanoma cells. No activation-related mutations in exon 11 of the KIT gene were found. On the contrary, expression of the stem cell growth factor (c-kit ligand) was detected in all three uveal melanoma cell lines, suggesting the presence of autocrine (paracrine) stimulation pathways. Treatment of uveal melanoma cell lines with STI571, which blocks c-kit autophosp

In [6]:
from typing import Any
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState


class State(AgentState):
    context: list[Document]


class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State

    def __init__(self, vector_store):
        self.vector_store = vector_store

    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state["messages"][-1] # get the user input query
        retrieved_docs = self.vector_store.similarity_search(last_message.text)  # search for documents

        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)  

        augmented_message_content = (f"""
        Answer the medical question using the context below.
        Respond with exactly one word: Yes or No.
        Context:
        {docs_content}
        
        Question:
        {last_message.text}""".strip())
        return {
            "messages": [last_message.model_copy(update={"content": augmented_message_content})],
            "context": retrieved_docs,
        }

In [7]:
from langchain.agents import create_agent

middleware = RetrieveDocumentsMiddleware(vectordb)
agent_rag = create_agent(model="gpt-4o-mini", tools=[], middleware=[middleware], system_prompt="Answer the medical question with exactly one word: Yes or No.")


your_query = questions.iloc[0]["question"]

answer = agent_rag.invoke({"messages": [{"role": "user", "content": your_query}]}, stream_mode="values")
# for step in agent_rag.stream({"messages": [{"role": "user", "content": your_query}]}, stream_mode="values"):
#     step["messages"][-1].pretty_print()

print(answer["messages"][-1].text.lower())
print(questions.iloc[0]["gold_label"])

yes
yes


In [8]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(agent, data):
    predictions = [agent.invoke({"messages": [{"role": "user", "content": question}]})["messages"][-1].text.strip().lower().strip(".,:;!*") for question in data["question"]]
    valid = [index for index, prediction in enumerate(predictions) if prediction in {"yes", "no"}]
    return (
        accuracy_score(data.iloc[valid]["gold_label"], [predictions[index] for index in valid]),
        f1_score(data.iloc[valid]["gold_label"], [predictions[index] for index in valid], pos_label="yes"),
        len(valid) / len(data)
    )


print("RAG accuracy and valid rate:", evaluate(agent_rag, questions.head(20)))

agent = create_agent(model="gpt-4o-mini", tools=[], middleware=[], system_prompt="Answer the medical question with exactly one word: Yes or No.")
print("NO RAG accuracy and valid rate:", evaluate(agent, questions.head(20)))

RAG accuracy and valid rate: (0.85, 0.88, 1.0)
NO RAG accuracy and valid rate: (0.65, 0.7586206896551724, 1.0)
